# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all entities by their `@id` and following best practices for FAIR/Croissant data.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and inspect dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")


## 2. Data Overview
List available record sets in the dataset and show their `@id`s and field composition. This helps you choose which record sets to explore.


In [ ]:
# Show all Record Sets by their @id
print("Record sets in the dataset:\n")
for recordset in dataset.record_sets:
    print(f"- @id: {recordset.id}, name: {recordset.name}, description: {getattr(recordset, 'description', '')}")
    # List the fields in this record set, referencing their @id as well
    fields = getattr(recordset, 'fields', [])
    if fields:
        for field in fields:
            print(f"    Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print("")

## 3. Data Extraction
Extract all data from each record set, referencing them by `@id`. Below, we build a dictionary of DataFrames, one per record set.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Selected record sets: {record_set_ids}\n")

# Load all record sets into separate DataFrames by @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set: {record_set_id} | Number of rows: {len(dataframes[record_set_id])} | Columns: {dataframes[record_set_id].columns.tolist()}")

# If there's at least one record set, show head of the first one
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"Columns for {main_rs}: {dataframes[main_rs].columns.tolist()}")
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
This section selects a numeric field from a record set, extracts and analyzes it, normalizes its values, and groups by a categorical field—all field and record set references are via their `@id` for programmatic reproducibility.


In [ ]:
# For demonstration, select the first record set and one numeric field (by @id)
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    
    # Try to find a numeric field
    rs_meta = next((r for r in dataset.record_sets if r.id == rs_id), None)
    numeric_fields = [f.id for f in getattr(rs_meta, 'fields', []) if f.data_type in ["Number", "Float", "Integer"]]
    group_fields = [f.id for f in getattr(rs_meta, 'fields', []) if f.data_type in ["Text", "String"]]
    
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        # Drop NA for this field and ensure numeric
        df = df.copy()
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Attempt grouping by first text/string field
        if group_fields:
            group_field = group_fields[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
                print(f"Grouped means by {group_field}:")
                print(grouped_df.head())
    else:
        print("No numeric field found in this record set.")

## 5. Visualization
Visualize the distribution of a numeric field if present and sufficient records exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram plot for the numeric field selected above
if record_set_ids and numeric_fields:
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, ax=ax, kde=True)
    ax.set_title(f"Distribution of {numeric_field_id} in {rs_id}")
    ax.set_xlabel(numeric_field_id)
    ax.set_ylabel("Count")
    plt.show()
else:
    print("No numeric fields found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load a FAIR-compliant Croissant dataset via `mlcroissant`, enumerate record sets and fields by their `@id`, extract and analyze data, and produce basic visualizations. For more detailed exploration, consult the field and record set metadata via their `@id` using `mlcroissant`. The FAIR^2 dataset makes advanced proteomics data available for downstream machine learning and FAIR data sharing.